[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wildertrek/catcher-in-the-cache/blob/main/notebooks/09_catcher_in_the_cache.ipynb)

# 09: The Catcher in the Cache (interactive view)

> *"People always think something's all true."* ~ Holden Caulfield, *The Catcher in the Rye*

This notebook visualizes the **activation probe** behind the paper's title finding: when frontier
LLMs rate the personality of canonical literary characters, they are largely **retrieving a
memorized character prior, not measuring traits from the text**. The behavioral falsifier shows
the Honesty-Humility (H) and HEXACO-Agreeableness (A_HEX) fusion collapses on characters absent
from training. The Catcher pulls that claim down to **model internals**: we probe hidden-state
activations across layer depth and watch the H/A_HEX trait subspace get **re-fused in the deep
layers**, the layer-by-layer signature of the cache.

All data is precomputed from saved probe summaries (12 open-weight raters, 7-72B, 60 characters).
Nothing here re-runs a model, so it runs anywhere in a few seconds.


## Setup

**In Colab, run the cell below first.** It clones the companion repository and installs dependencies (~30 s), then changes into `notebooks/` so the relative paths in this notebook resolve. Run locally, the cell is a no-op.

In [1]:
# Colab setup: clone the companion repo + install dependencies (no-op when run locally).
import sys, os, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("catcher-in-the-cache"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Wildertrek/catcher-in-the-cache.git"], check=True)
    os.chdir("catcher-in-the-cache/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())

In [2]:
%pip install -q plotly pandas numpy
import json, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prefer the live repo artifact; fall back to the embedded copy so this runs in a bare Colab.
EMBEDDED = r'''{
 "advanced": [
  {
   "hf_id": "Qwen/Qwen2.5-72B-Instruct",
   "family": "Qwen",
   "name": "Qwen2.5-72B-Instruct",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 10,
     "r_HA": 0.8304,
     "acc_H": 0.6167,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 20,
     "r_HA": 0.6419,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 40,
     "r_HA": -0.0769,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 60,
     "r_HA": 0.1755,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 79,
     "r_HA": -0.2219,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.7
    }
   ]
  },
  {
   "hf_id": "Qwen/Qwen2.5-7B-Instruct",
   "family": "Qwen",
   "name": "Qwen2.5-7B-Instruct",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 4,
     "r_HA": 0.7952,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 8,
     "r_HA": 0.7547,
     "acc_H": 0.65,
     "acc_A_HEX": 0.6
    },
    {
     "layer": 12,
     "r_HA": 0.5072,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.5667
    },
    {
     "layer": 16,
     "r_HA": 0.59,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 20,
     "r_HA": 0.5442,
     "acc_H": 0.75,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 24,
     "r_HA": 0.4114,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 28,
     "r_HA": 0.196,
     "acc_H": 0.65,
     "acc_A_HEX": 0.6833
    }
   ]
  },
  {
   "hf_id": "Qwen/Qwen2.5-7B",
   "family": "Qwen",
   "name": "Qwen2.5-7B",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 8,
     "r_HA": 0.7074,
     "acc_H": 0.6,
     "acc_A_HEX": 0.5833
    },
    {
     "layer": 16,
     "r_HA": 0.4873,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.65
    },
    {
     "layer": 24,
     "r_HA": 0.5142,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.7
    },
    {
     "layer": 28,
     "r_HA": 0.5288,
     "acc_H": 0.7,
     "acc_A_HEX": 0.7167
    }
   ]
  },
  {
   "hf_id": "google/gemma-2-9b-it",
   "family": "Gemma",
   "name": "gemma-2-9b-it",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 4,
     "r_HA": 0.7995,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.5833
    },
    {
     "layer": 8,
     "r_HA": 0.6856,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 12,
     "r_HA": 0.3923,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.7167
    },
    {
     "layer": 16,
     "r_HA": 0.2708,
     "acc_H": 0.75,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 20,
     "r_HA": 0.4317,
     "acc_H": 0.7,
     "acc_A_HEX": 0.7
    },
    {
     "layer": 24,
     "r_HA": 0.1667,
     "acc_H": 0.75,
     "acc_A_HEX": 0.65
    },
    {
     "layer": 28,
     "r_HA": 0.213,
     "acc_H": 0.75,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 31,
     "r_HA": -0.1101,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6
    }
   ]
  },
  {
   "hf_id": "google/gemma-2-9b",
   "family": "Gemma",
   "name": "gemma-2-9b",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 4,
     "r_HA": 0.8336,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.5833
    },
    {
     "layer": 8,
     "r_HA": 0.5627,
     "acc_H": 0.75,
     "acc_A_HEX": 0.6
    },
    {
     "layer": 12,
     "r_HA": 0.1125,
     "acc_H": 0.7667,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 16,
     "r_HA": 0.4332,
     "acc_H": 0.8,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 20,
     "r_HA": 0.5025,
     "acc_H": 0.8,
     "acc_A_HEX": 0.65
    },
    {
     "layer": 24,
     "r_HA": 0.5378,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 28,
     "r_HA": -0.0064,
     "acc_H": 0.7667,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 31,
     "r_HA": 0.0425,
     "acc_H": 0.7667,
     "acc_A_HEX": 0.6667
    }
   ]
  },
  {
   "hf_id": "meta-llama/Llama-3.1-70B-Instruct",
   "family": "Llama",
   "name": "Llama-3.1-70B-Instruct",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 10,
     "r_HA": 0.7264,
     "acc_H": 0.55,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 20,
     "r_HA": 0.7626,
     "acc_H": 0.5833,
     "acc_A_HEX": 0.5833
    },
    {
     "layer": 40,
     "r_HA": 0.6052,
     "acc_H": 0.75,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 60,
     "r_HA": 0.2802,
     "acc_H": 0.75,
     "acc_A_HEX": 0.6
    },
    {
     "layer": 79,
     "r_HA": 0.1695,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.65
    }
   ]
  },
  {
   "hf_id": "meta-llama/Llama-3.1-70B",
   "family": "Llama",
   "name": "Llama-3.1-70B",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 10,
     "r_HA": 0.9063,
     "acc_H": 0.5333,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 20,
     "r_HA": 0.786,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 40,
     "r_HA": 0.5951,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 60,
     "r_HA": 0.3824,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 79,
     "r_HA": 0.3738,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6167
    }
   ]
  },
  {
   "hf_id": "meta-llama/Llama-3.1-8B-Instruct",
   "family": "Llama",
   "name": "Llama-3.1-8B-Instruct",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 4,
     "r_HA": 0.6352,
     "acc_H": 0.55,
     "acc_A_HEX": 0.6
    },
    {
     "layer": 8,
     "r_HA": 0.7328,
     "acc_H": 0.55,
     "acc_A_HEX": 0.6
    },
    {
     "layer": 12,
     "r_HA": 0.7097,
     "acc_H": 0.6333,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 16,
     "r_HA": 0.6635,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 20,
     "r_HA": 0.5473,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 24,
     "r_HA": 0.4684,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 28,
     "r_HA": 0.6522,
     "acc_H": 0.7667,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 31,
     "r_HA": 0.6967,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.75
    }
   ]
  },
  {
   "hf_id": "meta-llama/Llama-3.1-8B",
   "family": "Llama",
   "name": "Llama-3.1-8B",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 4,
     "r_HA": 0.6865,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5833
    },
    {
     "layer": 8,
     "r_HA": 0.7386,
     "acc_H": 0.55,
     "acc_A_HEX": 0.6
    },
    {
     "layer": 12,
     "r_HA": 0.814,
     "acc_H": 0.6333,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 16,
     "r_HA": 0.8167,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 20,
     "r_HA": 0.7017,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 24,
     "r_HA": 0.7221,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 28,
     "r_HA": 0.7092,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.7167
    },
    {
     "layer": 31,
     "r_HA": 0.5039,
     "acc_H": 0.75,
     "acc_A_HEX": 0.7833
    }
   ]
  },
  {
   "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
   "family": "Mistral",
   "name": "Mistral-7B-Instruct-v0.3",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 4,
     "r_HA": 0.6749,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5
    },
    {
     "layer": 8,
     "r_HA": 0.6698,
     "acc_H": 0.5833,
     "acc_A_HEX": 0.5667
    },
    {
     "layer": 12,
     "r_HA": 0.7702,
     "acc_H": 0.55,
     "acc_A_HEX": 0.5833
    },
    {
     "layer": 16,
     "r_HA": 0.7762,
     "acc_H": 0.6167,
     "acc_A_HEX": 0.6167
    },
    {
     "layer": 20,
     "r_HA": 0.6376,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 24,
     "r_HA": 0.5863,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 28,
     "r_HA": 0.5649,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6833
    },
    {
     "layer": 31,
     "r_HA": 0.31,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.7
    }
   ]
  },
  {
   "hf_id": "mistralai/Mistral-7B-v0.3",
   "family": "Mistral",
   "name": "Mistral-7B-v0.3",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 4,
     "r_HA": 0.668,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.4833
    },
    {
     "layer": 8,
     "r_HA": 0.6497,
     "acc_H": 0.5833,
     "acc_A_HEX": 0.55
    },
    {
     "layer": 12,
     "r_HA": 0.7654,
     "acc_H": 0.5833,
     "acc_A_HEX": 0.5667
    },
    {
     "layer": 16,
     "r_HA": 0.7712,
     "acc_H": 0.6333,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 20,
     "r_HA": 0.6956,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6333
    },
    {
     "layer": 24,
     "r_HA": 0.5858,
     "acc_H": 0.7,
     "acc_A_HEX": 0.65
    },
    {
     "layer": 28,
     "r_HA": 0.5755,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.65
    },
    {
     "layer": 31,
     "r_HA": 0.3643,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6667
    }
   ]
  },
  {
   "hf_id": "mistralai/Mixtral-8x22B-Instruct-v0.1",
   "family": "Mixtral (MoE)",
   "name": "Mixtral-8x22B-Instruct-v0.1",
   "rating_time_r_HA": 0.7392,
   "per_layer": [
    {
     "layer": 5,
     "r_HA": 0.6782,
     "acc_H": 0.5333,
     "acc_A_HEX": 0.5333
    },
    {
     "layer": 14,
     "r_HA": 0.6402,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6
    },
    {
     "layer": 28,
     "r_HA": 0.6472,
     "acc_H": 0.8,
     "acc_A_HEX": 0.6667
    },
    {
     "layer": 42,
     "r_HA": -0.0738,
     "acc_H": 0.75,
     "acc_A_HEX": 0.7
    },
    {
     "layer": 55,
     "r_HA": 0.4465,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6667
    }
   ]
  }
 ],
 "small": [
  {
   "hf_id": "Qwen/Qwen2.5-7B-Instruct",
   "family": "Qwen",
   "name": "Qwen2.5-7B-Instruct",
   "tuned": true,
   "per_layer": [
    {
     "layer": 4,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6,
     "entanglement_gap": 0.05
    },
    {
     "layer": 8,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6333,
     "entanglement_gap": 0.0944
    },
    {
     "layer": 12,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6,
     "entanglement_gap": 0.08
    },
    {
     "layer": 16,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6333,
     "entanglement_gap": 0.0567
    },
    {
     "layer": 20,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.0444
    },
    {
     "layer": 24,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0225
    },
    {
     "layer": 28,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.7,
     "entanglement_gap": -0.0333
    }
   ]
  },
  {
   "hf_id": "Qwen/Qwen2.5-7B",
   "family": "Qwen",
   "name": "Qwen2.5-7B",
   "tuned": false,
   "per_layer": [
    {
     "layer": 8,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0725
    },
    {
     "layer": 16,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6333,
     "entanglement_gap": 0.09
    },
    {
     "layer": 24,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": -0.0222
    },
    {
     "layer": 28,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.7,
     "entanglement_gap": -0.0283
    }
   ]
  },
  {
   "hf_id": "google/gemma-2-9b-it",
   "family": "Gemma",
   "name": "gemma-2-9b-it",
   "tuned": true,
   "per_layer": [
    {
     "layer": 4,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": -0.0397
    },
    {
     "layer": 8,
     "acc_H": 0.7333,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": -0.0011
    },
    {
     "layer": 12,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.1111
    },
    {
     "layer": 16,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": 0.0217
    },
    {
     "layer": 20,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0392
    },
    {
     "layer": 24,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6,
     "entanglement_gap": 0.0967
    },
    {
     "layer": 28,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6167,
     "entanglement_gap": 0.0414
    },
    {
     "layer": 31,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.0556
    }
   ]
  },
  {
   "hf_id": "google/gemma-2-9b",
   "family": "Gemma",
   "name": "gemma-2-9b",
   "tuned": false,
   "per_layer": [
    {
     "layer": 4,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6333,
     "entanglement_gap": 0.0233
    },
    {
     "layer": 8,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": -0.0231
    },
    {
     "layer": 12,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0008
    },
    {
     "layer": 16,
     "acc_H": 0.7667,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": 0.0094
    },
    {
     "layer": 20,
     "acc_H": 0.7167,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": 0.0269
    },
    {
     "layer": 24,
     "acc_H": 0.7,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0283
    },
    {
     "layer": 28,
     "acc_H": 0.75,
     "acc_A_HEX": 0.7,
     "entanglement_gap": -0.0583
    },
    {
     "layer": 31,
     "acc_H": 0.75,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": -0.0667
    }
   ]
  },
  {
   "hf_id": "meta-llama/Llama-3.1-8B-Instruct",
   "family": "Llama",
   "name": "Llama-3.1-8B-Instruct",
   "tuned": true,
   "per_layer": [
    {
     "layer": 4,
     "acc_H": 0.55,
     "acc_A_HEX": 0.6,
     "entanglement_gap": 0.07
    },
    {
     "layer": 8,
     "acc_H": 0.5333,
     "acc_A_HEX": 0.6,
     "entanglement_gap": 0.1467
    },
    {
     "layer": 12,
     "acc_H": 0.5833,
     "acc_A_HEX": 0.6167,
     "entanglement_gap": 0.1236
    },
    {
     "layer": 16,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": 0.0778
    },
    {
     "layer": 20,
     "acc_H": 0.7,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.0667
    },
    {
     "layer": 24,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0167
    },
    {
     "layer": 28,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0667
    },
    {
     "layer": 31,
     "acc_H": 0.6333,
     "acc_A_HEX": 0.7167,
     "entanglement_gap": 0.0794
    }
   ]
  },
  {
   "hf_id": "meta-llama/Llama-3.1-8B",
   "family": "Llama",
   "name": "Llama-3.1-8B",
   "tuned": false,
   "per_layer": [
    {
     "layer": 4,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5667,
     "entanglement_gap": 0.0789
    },
    {
     "layer": 8,
     "acc_H": 0.5333,
     "acc_A_HEX": 0.6167,
     "entanglement_gap": 0.1211
    },
    {
     "layer": 12,
     "acc_H": 0.6333,
     "acc_A_HEX": 0.6167,
     "entanglement_gap": 0.1261
    },
    {
     "layer": 16,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.1333
    },
    {
     "layer": 20,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.0278
    },
    {
     "layer": 24,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.0667
    },
    {
     "layer": 28,
     "acc_H": 0.65,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.05
    },
    {
     "layer": 31,
     "acc_H": 0.7,
     "acc_A_HEX": 0.75,
     "entanglement_gap": 0.0083
    }
   ]
  },
  {
   "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
   "family": "Mistral",
   "name": "Mistral-7B-Instruct-v0.3",
   "tuned": true,
   "per_layer": [
    {
     "layer": 4,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5,
     "entanglement_gap": 0.15
    },
    {
     "layer": 8,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5333,
     "entanglement_gap": 0.1144
    },
    {
     "layer": 12,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5833,
     "entanglement_gap": 0.0861
    },
    {
     "layer": 16,
     "acc_H": 0.6,
     "acc_A_HEX": 0.6167,
     "entanglement_gap": 0.13
    },
    {
     "layer": 20,
     "acc_H": 0.6167,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": 0.1286
    },
    {
     "layer": 24,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.7,
     "entanglement_gap": 0.0217
    },
    {
     "layer": 28,
     "acc_H": 0.65,
     "acc_A_HEX": 0.6833,
     "entanglement_gap": 0.0558
    },
    {
     "layer": 31,
     "acc_H": 0.6833,
     "acc_A_HEX": 0.7167,
     "entanglement_gap": 0.0436
    }
   ]
  },
  {
   "hf_id": "mistralai/Mistral-7B-v0.3",
   "family": "Mistral",
   "name": "Mistral-7B-v0.3",
   "tuned": false,
   "per_layer": [
    {
     "layer": 4,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.4833,
     "entanglement_gap": 0.1594
    },
    {
     "layer": 8,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5333,
     "entanglement_gap": 0.1144
    },
    {
     "layer": 12,
     "acc_H": 0.5667,
     "acc_A_HEX": 0.5667,
     "entanglement_gap": 0.1122
    },
    {
     "layer": 16,
     "acc_H": 0.6333,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.155
    },
    {
     "layer": 20,
     "acc_H": 0.65,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.0833
    },
    {
     "layer": 24,
     "acc_H": 0.6667,
     "acc_A_HEX": 0.6667,
     "entanglement_gap": 0.0556
    },
    {
     "layer": 28,
     "acc_H": 0.6167,
     "acc_A_HEX": 0.6333,
     "entanglement_gap": 0.0761
    },
    {
     "layer": 31,
     "acc_H": 0.6333,
     "acc_A_HEX": 0.65,
     "entanglement_gap": 0.105
    }
   ]
  }
 ],
 "pool": [
  {
   "hf_id": "meta-llama/Llama-3.1-70B-Instruct",
   "name": "Llama-3.1-70B-Instruct",
   "arch": "dense",
   "aggregate": {
    "mean_pool_last_n": {
     "mean_r": 0.4888,
     "depth_delta": 0.7302,
     "delta_vs_rating_time": -0.2505
    },
    "last_token": {
     "mean_r": 0.5584,
     "depth_delta": 0.1586,
     "delta_vs_rating_time": -0.1808
    },
    "first_n_mean": {
     "mean_r": 0.6285,
     "depth_delta": 0.0266,
     "delta_vs_rating_time": -0.1107
    }
   }
  },
  {
   "hf_id": "mistralai/Mixtral-8x22B-Instruct-v0.1",
   "name": "Mixtral-8x22B-Instruct-v0.1",
   "arch": "sparse-MoE",
   "aggregate": {
    "mean_pool_last_n": {
     "mean_r": 0.2843,
     "depth_delta": 0.9682,
     "delta_vs_rating_time": -0.4549
    },
    "last_token": {
     "mean_r": 0.6245,
     "depth_delta": 0.2927,
     "delta_vs_rating_time": -0.1148
    },
    "first_n_mean": {
     "mean_r": 0.5034,
     "depth_delta": 0.3258,
     "delta_vs_rating_time": -0.2358
    }
   }
  }
 ],
 "substrate": {
  "canonical_r": 0.74,
  "synth_r": 0.29,
  "delta": -0.451,
  "p_bh": 7.5e-08,
  "rating_time_r_HA": 0.7392
 },
 "meta": {
  "n_target_chars": 60,
  "source": "v6_advanced_probe_results.json + v6_probe_results.json + v6_pool_ablation_*.json",
  "note": "Consolidated plotting fields for the Catcher-in-the-Cache viz notebook. Generated from saved probe summaries; no recomputation."
 }
}'''
try:
    from pathlib import Path
    p = Path('paper_artifacts/pivot6_hexaco_atlas/v6_activation_probe/catcher_viz_data.json')
    if not p.exists():
        p = Path('../paper_artifacts/pivot6_hexaco_atlas/v6_activation_probe/catcher_viz_data.json')
    DATA = json.loads(p.read_text()) if p.exists() else json.loads(EMBEDDED)
    print('Loaded live artifact' if p.exists() else 'Loaded embedded data')
except Exception:
    DATA = json.loads(EMBEDDED); print('Loaded embedded data')

FAMILY_COLORS = {'Qwen':'#6C5CE7','Llama':'#0984E3','Gemma':'#00B894',
                 'Mistral':'#E17055','Mixtral (MoE)':'#D63031'}
TEMPLATE = 'plotly_white'
print(f"{DATA['meta']['n_target_chars']} characters | {len(DATA['advanced'])} models with depth trajectories")


Note: you may need to restart the kernel to use updated packages.


Loaded live artifact
60 characters | 12 models with depth trajectories


## 1. The money shot: trait fusion vs layer depth

For each frontier (70B+) rater we plot the correlation between the probe's predicted H and predicted
A_HEX, `r(pred-H, pred-A_HEX)`, at each probed layer (normalized to depth fraction). The dashed line
is the **rating-time** fusion (~0.74), what the model outputs when you just ask it. If the model
were *measuring*, the trait subspace would stay separable through depth. Instead the correlation
starts high in early layers and **collapses (even flips negative) in the deep layers**: the deep
layers re-fusion H and A_HEX, the cache's representational fingerprint.


In [3]:
frontier = ['Qwen/Qwen2.5-72B-Instruct','meta-llama/Llama-3.1-70B-Instruct',
            'mistralai/Mixtral-8x22B-Instruct-v0.1']
fig = go.Figure()
for a in DATA['advanced']:
    if a['hf_id'] not in frontier: continue
    layers = [p['layer'] for p in a['per_layer']]
    maxL = max(layers) or 1
    xs = [l/maxL for l in layers]
    ys = [p['r_HA'] for p in a['per_layer']]
    d0, d1 = ys[0], ys[-1]
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers', name=f"{a['name']}  (Δ={d1-d0:+.2f})",
        line=dict(color=FAMILY_COLORS.get(a['family'],'#555'), width=3), marker=dict(size=9),
        customdata=layers,
        hovertemplate='%{fullData.name}<br>layer %{customdata} (depth %{x:.0%})<br>r(H,A_HEX)=%{y:+.3f}<extra></extra>'))
rt = DATA['substrate']['rating_time_r_HA']
fig.add_hline(y=rt, line_dash='dash', line_color='#b2bec3',
              annotation_text=f'rating-time fusion r={rt:.2f}', annotation_position='top left')
fig.add_hline(y=0, line_color='#dfe6e9')
fig.update_layout(template=TEMPLATE, height=520,
    title='Trait fusion collapses with depth (3 independent 70B+ families)',
    xaxis_title='layer depth (fraction of model depth)', yaxis_title='r(predicted H, predicted A_HEX)',
    legend=dict(orientation='h', y=-0.2), xaxis=dict(tickformat='.0%'))
fig.show()


## 2. Entanglement gap across the smaller base vs instruct raters

The `entanglement_gap` is how much better a *joint* H+A_HEX probe does than independent probes, a
direct measure of how fused the two traits are in that layer's representation. Hover any cell.
The effect is strongest at 70B+ (Figure 1); at 7-9B it is present but noisier.


In [4]:
rows = sorted(DATA['small'], key=lambda m:(m['family'], m['tuned']))
layer_ids = sorted({p['layer'] for m in rows for p in m['per_layer']})
z, ylabels = [], []
for m in rows:
    gap_by_layer = {p['layer']:p['entanglement_gap'] for p in m['per_layer']}
    z.append([gap_by_layer.get(l, None) for l in layer_ids])
    ylabels.append(f"{m['name']} {'(instruct)' if m['tuned'] else '(base)'}")
fig = go.Figure(go.Heatmap(z=z, x=[f'L{l}' for l in layer_ids], y=ylabels,
    colorscale='RdBu_r', zmid=0, colorbar=dict(title='entangle<br>gap'),
    hovertemplate='%{y}<br>%{x}<br>entanglement gap=%{z:+.3f}<extra></extra>'))
fig.update_layout(template=TEMPLATE, height=480,
    title='Joint-minus-independent probe gap (fusion) by layer, 7-9B raters',
    xaxis_title='probed layer', yaxis_title='')
fig.show()


## 3. Architecture contrast: dense vs sparse-MoE pool ablation

Where does the trait subspace live? We re-probe three pooling strategies. `layer_depth_delta`
(early-minus-late fusion) is large for the **dense** Llama under mean-pooling (it localizes the
subspace), while the **sparse-MoE** Mixtral spreads the effect across pooling strategies. Same
phenomenon, different architectural footprint.


In [5]:
pools = DATA['pool'][0]['aggregate'].keys()
fig = go.Figure()
for pdat in DATA['pool']:
    fig.add_trace(go.Bar(name=f"{pdat['name']} [{pdat['arch']}]",
        x=list(pdat['aggregate'].keys()),
        y=[pdat['aggregate'][p]['depth_delta'] for p in pdat['aggregate']],
        hovertemplate='%{fullData.name}<br>%{x}<br>early-late depth delta=%{y:+.3f}<extra></extra>'))
fig.update_layout(template=TEMPLATE, height=460, barmode='group',
    title='Layer-depth fusion delta by pooling strategy (dense vs sparse-MoE)',
    xaxis_title='pooling strategy', yaxis_title='early-minus-late fusion delta')
fig.show()


## 4. The tie-in: internals meet behavior (the substrate falsifier)

The Catcher (internals, Figures 1-3) explains the **behavioral** falsifier. When raters score
canonical characters they are in the cache, within-rater |r(H, A_HEX)| ~ 0.74. On 20 **synthetic**
characters absent from any training corpus, same models, same probe, it drops to ~0.29
(Δ=-0.451, paired Wilcoxon, p_BH=7.5e-8). Take away the cached prior and the fusion goes with it.


In [6]:
s = DATA['substrate']
fig = go.Figure(go.Bar(
    x=['canonical<br>(in cache)','synthetic<br>(out of cache)'], y=[s['canonical_r'], s['synth_r']],
    marker_color=['#0984E3','#D63031'], text=[f"|r|={s['canonical_r']}", f"|r|={s['synth_r']}"],
    textposition='outside', hovertemplate='%{x}<br>|r(H,A_HEX)|=%{y}<extra></extra>'))
fig.add_annotation(x=0.5, y=max(s['canonical_r'],s['synth_r'])*0.95, showarrow=False,
    text=f"Δ = {s['delta']}   (p_BH = {s['p_bh']:.1e})", font=dict(size=14))
fig.update_layout(template=TEMPLATE, height=460, yaxis_range=[0,0.9],
    title='Fusion survives in the cache, collapses out of it',
    yaxis_title='within-rater |r(H, A_HEX)|', showlegend=False)
fig.show()


> **Reading the endpoints.** The bars show *absolute* within-rater |r| (canonical ≈ 0.75, synthetic ≈ 0.30), so Δ = −0.45 is a magnitude drop. The paper's concept figure uses the *signed* residual instead (canonical +0.75, synthetic **+0.23**): the fusion doesn't just weaken, it collapses toward zero. Same 25-rater result, viewed as |r| (magnitude) vs signed (direction).

## Reading

- **Figure 1**: in three independent 70B+ families the predicted-trait fusion starts high and
  collapses with depth (Qwen-72B Δ=-1.05, the cleanest), diverging from the steady ~0.74 rating-time
  output. The model's *surface* says one thing; its *deep layers* do another.
- **Figures 2-3**: the effect is representational (entanglement gap) and has an architecture-dependent
  footprint (dense localizes, MoE distributes).
- **Figure 4**: the internals match the behavior, removing the cached prior removes the fusion.

Holden's word for a fluent surface that masks a different mechanism is *phony*. That is the precise
sense in which an LLM personality rating is phony: it looks like measurement and is mostly retrieval.

Source artifacts: `paper_artifacts/pivot6_hexaco_atlas/v6_activation_probe/` (this notebook reads the
consolidated `catcher_viz_data.json`). Narrative companion: `CATCHER.md`.
